In [4]:
pip install anywidget ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 874.0/874.0 kB 730.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [anywidget]/2 [anywidget]
Note: you may need to restart the kernel to use updated packages.


In [4]:
%matplotlib notebook


In [5]:
import plotly.graph_objects as go
import pandas as pd
from ipywidgets import Button, VBox

# Store clicked points
clicked_points = []

# Create a Plotly FigureWidget
fig = go.FigureWidget(
    data=[go.Scatter(x=[], y=[], mode='markers+lines', marker=dict(color='red'))],
    layout=go.Layout(
        title="Click to add trajectory points",
        xaxis=dict(range=[-10, 10]),
        yaxis=dict(range=[-10, 10]),
    )
)

scatter = fig.data[0]

# Define click callback
def handle_click(trace, points, selector):
    for i in points.point_inds:
        # Add the clicked point
        x_new = trace.x[i]
        y_new = trace.y[i]
        clicked_points.append((x_new, y_new))

# But FigureWidget doesn't have raw click — workaround:
# We'll use a "click-data" handler by overlaying a dummy scatter
fig = go.FigureWidget()
scatter_points = go.Scatter(x=[], y=[], mode='markers+lines', marker=dict(color='red'))
background = go.Scatter(x=[-10, 10], y=[-10, 10], mode='markers', opacity=0)  # dummy background

fig.add_trace(scatter_points)
fig.add_trace(background)

fig.update_layout(
    title="Click anywhere to add trajectory points",
    xaxis=dict(range=[-10, 10]),
    yaxis=dict(range=[-10, 10]),
)

# Function to handle click events
def update_point(trace, points, selector):
    if points.xs and points.ys:
        x_new, y_new = points.xs[0], points.ys[0]
        clicked_points.append((x_new, y_new))
        with fig.batch_update():
            scatter_points.x += (x_new,)
            scatter_points.y += (y_new,)

background.on_click(update_point)

# Save button
save_btn = Button(description="Save to CSV")

def save_callback(b):
    df = pd.DataFrame(clicked_points, columns=["x", "y"])
    df.to_csv("trajectory.csv", index=False)
    print("Trajectory saved to trajectory.csv")

save_btn.on_click(save_callback)

VBox([fig, save_btn])


    'data': [{'marker': {'color': 'red'},
              'mode': 'markers+lines',…